In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
df_raw = pd.read_parquet("/home/tamer/projects/mlops-practitioner/data/raw/green_tripdata_2023-01.parquet")
df_raw.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68211 entries, 0 to 68210
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               68211 non-null  int64         
 1   lpep_pickup_datetime   68211 non-null  datetime64[us]
 2   lpep_dropoff_datetime  68211 non-null  datetime64[us]
 3   store_and_fwd_flag     63887 non-null  object        
 4   RatecodeID             63887 non-null  float64       
 5   PULocationID           68211 non-null  int64         
 6   DOLocationID           68211 non-null  int64         
 7   passenger_count        63887 non-null  float64       
 8   trip_distance          68211 non-null  float64       
 9   fare_amount            68211 non-null  float64       
 10  extra                  68211 non-null  float64       
 11  mta_tax                68211 non-null  float64       
 12  tip_amount             68211 non-null  float64       
 13  t

In [4]:
df_raw.size

1364220

In [5]:
df_raw.describe()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
count,68211.000000,68211,68211,63887.000000,68211.000000,68211.000000,63887.000000,68211.000000,68211.000000,68211.000000,68211.000000,68211.000000,68211.000000,68211.000000,68211.000000,63887.000000,63877.000000,63887.000000
mean,1.863028,2023-01-16 20:10:55.679523,2023-01-16 20:29:01.515767,1.117160,98.549735,138.429901,1.315870,8.114852,16.603545,0.825431,0.588340,2.139012,0.169627,0.951625,21.789377,1.373613,1.021933,0.726748
min,1.000000,2009-01-01 20:21:27,2009-01-02 11:07:31,1.000000,1.000000,1.000000,0.000000,0.000000,-70.000000,-2.500000,-0.500000,-10.500000,0.000000,-1.000000,-71.500000,1.000000,1.000000,-2.750000
25%,2.000000,2023-01-09 11:59:47.500000,2023-01-09 12:16:37.500000,1.000000,74.000000,74.000000,1.000000,1.110000,9.300000,0.000000,0.500000,0.000000,0.000000,1.000000,12.900000,1.000000,1.000000,0.000000
50%,2.000000,2023-01-17 08:40:42,2023-01-17 08:56:38,1.000000,75.000000,138.000000,1.000000,1.850000,13.500000,0.000000,0.500000,1.600000,0.000000,1.000000,17.900000,1.000000,1.000000,0.000000
75%,2.000000,2023-01-24 15:52:30,2023-01-24 16:06:56,1.000000,129.000000,219.000000,1.000000,3.210000,19.800000,1.000000,0.500000,3.330000,0.000000,1.000000,26.150000,2.000000,1.000000,2.750000
max,2.000000,2023-02-01 03:10:05,2023-02-01 17:27:05,99.000000,265.000000,265.000000,9.000000,120098.840000,490.000000,12.500000,2.750000,222.220000,36.050000,1.000000,491.000000,5.000000,2.000000,2.750000
std,0.343820,NaN,NaN,1.372913,61.244314,76.761311,0.979054,585.105955,13.470121,1.269904,0.385819,3.052710,1.099789,0.196875,15.457115,0.508039,0.146465,1.212836


In [6]:
df_raw["lpep_dropoff_datetime"][:5]

0   2023-01-01 00:37:11
1   2023-01-01 00:57:49
2   2023-01-01 00:41:32
3   2023-01-01 00:19:03
4   2023-01-01 00:39:02
Name: lpep_dropoff_datetime, dtype: datetime64[us]

In [7]:
df_raw["lpep_pickup_datetime"][:5]

0   2023-01-01 00:26:10
1   2023-01-01 00:51:03
2   2023-01-01 00:35:12
3   2023-01-01 00:13:14
4   2023-01-01 00:33:04
Name: lpep_pickup_datetime, dtype: datetime64[us]

In [8]:
df_raw["lpep_dropoff_datetime"][1] - df_raw["lpep_pickup_datetime"][1]

Timedelta('0 days 00:06:46')

In [9]:
# calculate the duration in minutes
df_raw['duration'] = df_raw.lpep_dropoff_datetime - df_raw.lpep_pickup_datetime
df_raw["duration"][:5]

0   0 days 00:11:01
1   0 days 00:06:46
2   0 days 00:06:20
3   0 days 00:05:49
4   0 days 00:05:58
Name: duration, dtype: timedelta64[us]

In [10]:
# convert the calculated duration from seconds to minutes
df_raw.duration.dt.total_seconds() / 60

0        11.016667
1         6.766667
2         6.333333
3         5.816667
4         5.966667
           ...    
68206    13.000000
68207     8.000000
68208    16.000000
68209    18.000000
68210    16.000000
Name: duration, Length: 68211, dtype: float64

In [11]:
df_raw.duration = df_raw.duration.dt.total_seconds() / 60


In [12]:
df_raw.duration

0        11.016667
1         6.766667
2         6.333333
3         5.816667
4         5.966667
           ...    
68206    13.000000
68207     8.000000
68208    16.000000
68209    18.000000
68210    16.000000
Name: duration, Length: 68211, dtype: float64

In [13]:
df_raw.duration.min()

np.float64(0.0)

In [14]:
df_raw.duration.max()

np.float64(1439.8833333333334)

In [15]:
# Filter realistic trip durations (e.g., 1 minute to 60 minutes)
df = df_raw[(df_raw.duration >= 1) & (df_raw.duration <= 60)].copy()

In [16]:
df_raw.size

1432431

In [17]:
df_raw["PULocationID"]

0        166
1         24
2        223
3         41
4         41
        ... 
68206     49
68207     10
68208     66
68209    225
68210    256
Name: PULocationID, Length: 68211, dtype: int64

In [18]:
df_raw.describe().T

,count,mean,min,25%,50%,75%,max,std
VendorID,68211.0,1.863028,1.0,2.0,2.0,2.0,2.0,0.34382
lpep_pickup_datetime,68211,2023-01-16 20:10:55.679523,2009-01-01 20:21:27,2023-01-09 11:59:47.500000,2023-01-17 08:40:42,2023-01-24 15:52:30,2023-02-01 03:10:05,NaN
lpep_dropoff_datetime,68211,2023-01-16 20:29:01.515767,2009-01-02 11:07:31,2023-01-09 12:16:37.500000,2023-01-17 08:56:38,2023-01-24 16:06:56,2023-02-01 17:27:05,NaN
RatecodeID,63887.0,1.11716,1.0,1.0,1.0,1.0,99.0,1.372913
PULocationID,68211.0,98.549735,1.0,74.0,75.0,129.0,265.0,61.244314
DOLocationID,68211.0,138.429901,1.0,74.0,138.0,219.0,265.0,76.761311
passenger_count,63887.0,1.31587,0.0,1.0,1.0,1.0,9.0,0.979054
trip_distance,68211.0,8.114852,0.0,1.11,1.85,3.21,120098.84,585.105955
fare_amount,68211.0,16.603545,-70.0,9.3,13.5,19.8,490.0,13.470121
extra,68211.0,0.825431,-2.5,0.0,0.0,1.0,12.5,1.269904


In [19]:
df = df_raw.copy()


In [20]:
df["duration_min"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60
print(df["duration_min"].describe(percentiles=[.01, .5, .99]).round(2).to_string())
print("\nnegative durations:", (df["duration_min"] < 0).sum())
print("longer than 3 hours:", (df["duration_min"] > 180).sum())

count    68211.00
mean        18.10
std         74.93
min          0.00
1%           0.08
50%         11.40
99%         58.11
max       1439.88

negative durations: 0
longer than 3 hours: 264


In [21]:
df.isna().sum()

VendorID                     0
lpep_pickup_datetime         0
lpep_dropoff_datetime        0
store_and_fwd_flag        4324
RatecodeID                4324
PULocationID                 0
DOLocationID                 0
passenger_count           4324
trip_distance                0
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
ehail_fee                68211
improvement_surcharge        0
total_amount                 0
payment_type              4324
trip_type                 4334
congestion_surcharge      4324
duration                     0
duration_min                 0
dtype: int64

In [22]:
df.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'duration', 'duration_min'],
      dtype='object')

In [23]:
drop_cols = ["store_and_fwd_flag", "RatecodeID","passenger_count","ehail_fee","payment_type","trip_type","congestion_surcharge"]
df = df.drop(columns=drop_cols)


In [24]:
df.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'PULocationID', 'DOLocationID', 'trip_distance', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'duration', 'duration_min'],
      dtype='object')

In [25]:
df[["duration","duration_min"]]

,duration,duration_min
0,11.016667,11.016667
1,6.766667,6.766667
2,6.333333,6.333333
3,5.816667,5.816667
4,5.966667,5.966667
...,...,...
68206,13.000000,13.000000
68207,8.000000,8.000000
68208,16.000000,16.000000
68209,18.000000,18.000000


In [26]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,duration,duration_min
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,166,143,2.58,14.9,1.0,0.5,4.03,0.0,1.0,24.18,11.016667,11.016667
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,24,43,1.81,10.7,1.0,0.5,2.64,0.0,1.0,15.84,6.766667,6.766667
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,223,179,0.00,7.2,1.0,0.5,1.94,0.0,1.0,11.64,6.333333,6.333333
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,41,238,1.30,6.5,0.5,1.5,1.70,0.0,1.0,10.20,5.816667,5.816667
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,41,74,1.10,6.0,0.5,1.5,0.00,0.0,1.0,8.00,5.966667,5.966667


In [27]:
dropped_cols = ["lpep_pickup_datetime","lpep_dropoff_datetime","PULocationID","DOLocationID"]
df.drop(dropped_cols, axis=1, inplace=True)

In [28]:
df.drop("VendorID", axis=1, inplace=True) 

In [29]:
df.columns

Index(['trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'improvement_surcharge', 'total_amount', 'duration',
       'duration_min'],
      dtype='object')

In [30]:
df = df.drop_duplicates()

In [31]:
# 7. rows without pickup/dropoff zone
df = df.dropna()

In [32]:
df.shape

(67446, 10)

In [33]:
df_raw.shape

(68211, 21)

In [34]:
import pathlib as Path

In [35]:
SEED = 42
ROOT = r"/home/tamer/projects/mlops-practitioner/notebooks/01_baseline.ipynb"
RAW_DATA = r"/home/tamer/projects/mlops-practitioner/data/raw/green_tripdata_2023-01.parquet"
CLEAN_FILE = r"/home/tamer/projects/mlops-practitioner/data/cleaned/green_tripdata_2023-01_clean.parquet"
PARAMS = {
    "seed": SEED,
    "min_duration_min": 1,      # shorter trips are meter glitches
    "max_duration_min": 60,     # longer trips are very rare
    "min_distance_mi": 0.1,
    "max_distance_mi": 30,
    "max_passengers": 6,
    "val_size": 0.15,
    "test_size": 0.15,
    "rare_category_min_freq": 30,
}

In [36]:
print(CLEAN_FILE)

/home/tamer/projects/mlops-practitioner/data/cleaned/green_tripdata_2023-01_clean.parquet


In [37]:
df.to_parquet(CLEAN_FILE, index=False)

In [38]:
df_model = pd.read_parquet(CLEAN_FILE)
df_model.head()

,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,duration,duration_min
0,2.58,14.9,1.0,0.5,4.03,0.0,1.0,24.18,11.016667,11.016667
1,1.81,10.7,1.0,0.5,2.64,0.0,1.0,15.84,6.766667,6.766667
2,0.00,7.2,1.0,0.5,1.94,0.0,1.0,11.64,6.333333,6.333333
3,1.30,6.5,0.5,1.5,1.70,0.0,1.0,10.20,5.816667,5.816667
4,1.10,6.0,0.5,1.5,0.00,0.0,1.0,8.00,5.966667,5.966667


In [39]:
df_model.dtypes

trip_distance            float64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
duration                 float64
duration_min             float64
dtype: object

In [40]:
X = df_model.drop("duration",axis=1)
X.head()

,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,duration_min
0,2.58,14.9,1.0,0.5,4.03,0.0,1.0,24.18,11.016667
1,1.81,10.7,1.0,0.5,2.64,0.0,1.0,15.84,6.766667
2,0.00,7.2,1.0,0.5,1.94,0.0,1.0,11.64,6.333333
3,1.30,6.5,0.5,1.5,1.70,0.0,1.0,10.20,5.816667
4,1.10,6.0,0.5,1.5,0.00,0.0,1.0,8.00,5.966667


In [41]:
y = df_model["duration"]
y

0        11.016667
1         6.766667
2         6.333333
3         5.816667
4         5.966667
           ...    
67441    13.000000
67442     8.000000
67443    16.000000
67444    18.000000
67445    16.000000
Name: duration, Length: 67446, dtype: float64

# SPlitting the DATA

In [42]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=PARAMS["test_size"], random_state=SEED)


In [43]:
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest, test_size=PARAMS["val_size"] / (1 - PARAMS["test_size"]), random_state=SEED)

print(f"train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}")

train=47,212  val=10,117  test=10,117


# Scalling

In [44]:
from sklearn.preprocessing import MinMaxScaler
# Scale features (drop ID)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reattach IDs with aligned indices
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)


X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)


# Training Random Forest 

In [45]:
params = {
"max_depth":70,
"random_state":42,
"n_estimators":30,
"n_jobs":-1,
"verbose":1,
"criterion":'absolute_error'
}

In [46]:
from sklearn.ensemble import RandomForestRegressor

regr = RandomForestRegressor(**params)
regr.fit(X_train_scaled_df, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:  9.1min finished


,n_estimators,30
,criterion,'absolute_error'
,max_depth,70
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [47]:
regr.get_metadata_routing()

{'fit': {'sample_weight': None}, 'score': {'sample_weight': None}}

In [48]:
y_pred = regr.predict(X_test_scaled_df)
print(y_pred)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


[15.6         9.2         6.2        ...  4.83333333 24.15
 22.88333333]


[Parallel(n_jobs=4)]: Done  30 out of  30 | elapsed:    0.1s finished


In [49]:
mse = mean_squared_error(y_test, y_pred)
print(mse)

0.1668505619132678


In [50]:
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

0.4084734531316176


In [51]:
from sklearn.metrics import (
    mean_squared_error, 
    root_mean_squared_error, 
    mean_absolute_error, 
    r2_score, 
    mean_absolute_percentage_error, 
    max_error
)

In [52]:
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
max_err = max_error(y_test, y_pred)

# MLFlow

In [54]:
import mlflow
# the Experiment name
mlflow.set_experiment("RandomForestRegressor Baseline")
# connect the ml flow with the ui url 
mlflow.set_tracking_uri("http://127.0.0.1:5000")

with mlflow.start_run():
    # Log the params of the model we used in training
    # 1. Log the model's parameters
    params = {
        "max_depth": 70,
        "random_state": 42,
        "n_estimators": 30,
        "n_jobs": -1,
        "verbose": 1,
        "criterion": "absolute_error"
    }
    mlflow.log_params(params)

    # Initialize and train the model using those params
    regr = RandomForestRegressor(**params)
    regr.fit(X_train_scaled_df, y_train)

    # Generate predictions for evaluation
    y_pred = regr.predict(X_test_scaled_df)

    # logging the metrices we used
    # first we want to calc the metrices 
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    max_err = max_error(y_test, y_pred)

    mlflow.log_metrics({
    "mse": mse,
    "rmse": rmse,
    "mae": mae,
    "r2_score": r2,
    "mape": mape,
    "max_error": max_err
    })

    mlflow.sklearn.log_model(
        regr,
        artifact_path="random_forest_model",

        skops_trusted_types=["sklearn.tree._tree.Tree"] # That tells mlflow that my model is secure to serialize it because mlflow not trust the pickle models
    )

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:  8.8min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  30 out of  30 | elapsed:    0.1s finished
2026/09/29 08:45:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/29 08:46:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run kindly-loon-627 at: http://127.0.0.1:5000/#/experiments/1/runs/a37a784cf1f647759338a3bde097d5f4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


# Try Multiple models

In [55]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
import mlflow

In [56]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("taxi_model_comparison")


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1790656150435, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1790656150435, lifecycle_stage='active', name='taxi_model_comparison', tags={}, trace_location=None, workspace='default'>

In [69]:
# 2. Define the models and their important parameters in dictionaries
# Format: (Model Name, Model Class, Parameter Dictionary)
models_to_try = [
    (
        "Linear_Regression", 
        LinearRegression, 
        {"fit_intercept": True}
    ),
    (
        "Decision_Tree", 
        DecisionTreeRegressor, 
        {"max_depth": 100, "min_samples_split": 5, "random_state": 42, "criterion": "absolute_error"}
    ),
    (
        "Random_Forest", 
        RandomForestRegressor, 
        {"max_depth": 50, "n_estimators": 30, "min_samples_split": 5, "random_state": 42, "n_jobs": -1}
    ),
    (
        "XGBoost", 
        XGBRegressor, 
        {"max_depth": 20, "n_estimators": 30, "learning_rate": 0.1, "random_state": 42, "n_jobs": -1}
    )
]

## Connect to dagshub 
#### for make my experiments puplic in the cloud so any one in my team be able to see my experiments

In [67]:
import dagshub
dagshub.init(repo_owner='tamer-elkoT', repo_name='mlops-practitioner', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b95cd9a6-a3aa-4b75-879a-82f9921ff3a0&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4f6684320e1b6de6af72b61f0b6796160764950f8293e01748ba283b752ccb9c




/usr/bin/xdg-open: 882: x-www-browser: not found
/usr/bin/xdg-open: 882: firefox: not found
/usr/bin/xdg-open: 882: iceweasel: not found


/usr/bin/xdg-open: 882: seamonkey: not found
/usr/bin/xdg-open: 882: mozilla: not found


...

/usr/bin/xdg-open: 882: google-chrome: not found
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found


/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening 'https://dagshub.com/login/oauth/authorize?state=b95cd9a6-a3aa-4b75-879a-82f9921ff3a0&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4f6684320e1b6de6af72b61f0b6796160764950f8293e01748ba283b752ccb9c'


Accessing as tamer-elkoT

Initialized MLflow to track repo "tamer-elkoT/mlops-practitioner"

Repository tamer-elkoT/mlops-practitioner initialized!

In [70]:

mlflow.set_experiment("NYC Ride Duration")

mlflow.set_tracking_uri("https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow")

# 3. Iterate through the models
for model_name, ModelClass, params in models_to_try:
    with mlflow.start_run(run_name=model_name):
        print(f"Training {model_name}...")
        
        # Log only the explicitly defined important parameters
        mlflow.log_params(params)
        
        # Initialize the model by unpacking the dictionary
        model = ModelClass(**params)
        
        # Track training time
        start_time = time.time()
        model.fit(X_train_scaled_df, y_train)
        training_time = time.time() - start_time
        
        # Generate predictions
        y_pred = model.predict(X_test_scaled_df)
        
        # Calculate metrics
        rmse = root_mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        
        # Log metrics
        mlflow.log_metrics({
            "rmse": rmse,
            "mae": mae,
            "training_time_seconds": training_time
        })
        
        # Save the model artifact based on its specific library requirements
        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(model, artifact_path="model")
        elif "Tree" in model_name or "Forest" in model_name:
            mlflow.sklearn.log_model(
                model, 
                artifact_path="model",
                skops_trusted_types=["sklearn.tree._tree.Tree"]
            )
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")
            
        print(f"Finished {model_name} | RMSE: {rmse:.4f} | MAE: {mae:.4f}\n")

2026/09/29 11:32:27 INFO mlflow.tracking.fluent: Experiment with name 'NYC Ride Duration' does not exist. Creating a new experiment.


Training Linear_Regression...


2026/09/29 11:32:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/29 11:32:39 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished Linear_Regression | RMSE: 0.0000 | MAE: 0.0000

🏃 View run Linear_Regression at: https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow/#/experiments/0/runs/cf150c342e0d423aabf3bbc7b2c69902
🧪 View experiment at: https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow/#/experiments/0
Training Decision_Tree...


2026/09/29 11:33:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/29 11:34:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished Decision_Tree | RMSE: 1.1215 | MAE: 0.0326

🏃 View run Decision_Tree at: https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow/#/experiments/0/runs/c1809e11379e4685a675959ff4f3a1e2
🧪 View experiment at: https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow/#/experiments/0
Training Random_Forest...


2026/09/29 11:34:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/29 11:34:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished Random_Forest | RMSE: 0.7442 | MAE: 0.0218

🏃 View run Random_Forest at: https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow/#/experiments/0/runs/2506ef4a2729474395b3251952f5d987
🧪 View experiment at: https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow/#/experiments/0
Training XGBoost...


2026/09/29 11:35:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/29 11:35:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished XGBoost | RMSE: 20.5375 | MAE: 1.3765

🏃 View run XGBoost at: https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow/#/experiments/0/runs/811556d8eb0846c3857dec2ed634e164
🧪 View experiment at: https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow/#/experiments/0


# Register the model

In [59]:
model_name = "RandomForest"
model_id = "87ffc073055c4d52abd3331920610f79"
model_uri = f"runs:/{model_id}/model"

with mlflow.start_run(run_id=model_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'RandomForest'.
2026/09/29 10:19:55 WARNING mlflow.tracking._model_registry.fluent: Run with id 87ffc073055c4d52abd3331920610f79 has no artifacts at artifact path 'model', registering model based on models:/m-1801b7e96a0c476a977844122aca5d78 instead
2026/09/29 10:19:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest, version 1
Created version '1' of model 'RandomForest'.


🏃 View run Random_Forest at: http://127.0.0.1:5000/#/experiments/2/runs/87ffc073055c4d52abd3331920610f79
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


# Load the Model

In [62]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.pyfunc.load_model(model_uri)


In [63]:
y_pred = loaded_model.predict(X_test)
y_pred[:5]

array([1436.63286574, 1436.86786574, 1439.3471627 , 1436.64184722,
       1436.64184722])

# Transition the model to production

In [64]:
current_modle_uri =f"models:/{model_name}/{model_version}"
production_model_name = "NYC-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_modle_uri, dst_name=production_model_name)



Successfully registered model 'NYC-prod'.
Copied version '1' of model 'RandomForest' to version '1' of model 'NYC-prod'.


<ModelVersion: aliases=[], creation_timestamp=1790667566598, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1790667566598, metrics=None, model_id=None, name='NYC-prod', params=None, run_id='87ffc073055c4d52abd3331920610f79', run_link='', source='models:/RandomForest/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [66]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.pyfunc.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([1436.63286574, 1436.86786574, 1439.3471627 , 1436.64184722])